# Truth playground (milestone 1)

This notebook takes ADS-B truth from a recording, cleans it into per-flight
segments with the truth builder, plots an hour of traffic, and checks it
against a second source for the same aircraft.

It runs as-is on the synthetic fixture in `tests/data/`. To use your own
data, set these environment variables before starting Jupyter (or edit the
next cell):

| Variable | Meaning |
| --- | --- |
| `TRACKING_SIM_RECORDING` | A recording (`.parquet`/`.jsonl` from `tracking-sim record sbs`) or captured SBS text (`.sbs`, `.sbs.gz`) |
| `TRACKING_SIM_ORIGIN` | `LAT,LON[,ALT_M]` of the local frame, e.g. your receiver |
| `TRACKING_SIM_WINDOW` | `START,END` in ISO 8601 UTC, e.g. `2026-11-01T23:00Z,2026-11-02T00:00Z` |
| `TRACKING_SIM_SECOND` | A second source for the same hour, recording or SBS, e.g. a database export (section 5) |
| `TRACKING_SIM_SBS_TZ` | Time zone of the SBS timestamps (Dump1090 writes host local time; default UTC) |

In [ ]:
import os
from dataclasses import replace
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tracking_sim.cli import parse_time
from tracking_sim.geo import LocalFrame
from tracking_sim.truth import TruthBuilderConfig, build_truth
from tracking_sim.truth.compare import assess_altitude_reference, compare_truth
from tracking_sim.truth.recorder import summarize
from tracking_sim.truth.sources.recording import RecordingSource
from tracking_sim.truth.sources.sbs import SbsSource

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RECORDING = os.environ.get("TRACKING_SIM_RECORDING") or str(
    ROOT / "tests/data/synthetic_1h.parquet"
)
ORIGIN = os.environ.get("TRACKING_SIM_ORIGIN")
WINDOW = os.environ.get("TRACKING_SIM_WINDOW")
SECOND = os.environ.get("TRACKING_SIM_SECOND")
SBS_TZ = os.environ.get("TRACKING_SIM_SBS_TZ", "UTC")
plt.rcParams["figure.dpi"] = 110


def open_source(path, receiver, altitude_field="alt_baro_ft"):
    # SBS text (as Dump1090 serves it) or a recording written by tracking-sim.
    if ".sbs" in Path(path).suffixes:
        return SbsSource(path=path, receiver=receiver, tz=SBS_TZ, altitude_field=altitude_field)
    return RecordingSource(path)


print("recording:", RECORDING)

## 1. Load the recording

Every source replays `TruthRecord`s in time order: `RecordingSource` for
recordings and `SbsSource` for captured Dump1090 SBS text. Records are
sparse: position, velocity and identity arrive as separate messages.

In [ ]:
source = open_source(RECORDING, "primary")
start, end = (parse_time(x) for x in WINDOW.split(",")) if WINDOW else (None, None)
records = list(source.replay(start, end))
if start is None:
    start = records[0].ts.replace(minute=0, second=0, microsecond=0)
    end = start + timedelta(hours=1)
    records = [r for r in records if start <= r.ts < end]
print("\n".join(summarize(records).lines()))

## 2. Build truth

The builder deduplicates messages heard by more than one receiver, splits
each aircraft into segments on a 60 s gap, rejects bad-decode jumps with a
speed gate, merges altitude and velocity onto positions, converts to a local
ENU frame and fits a smoothing spline so `state_at(t)` works at any time
inside a segment.

In [ ]:
if ORIGIN:
    frame = LocalFrame(*[float(x) for x in ORIGIN.split(",")])
else:  # the centroid of the positions
    lats = [r.lat for r in records if r.lat is not None]
    lons = [r.lon for r in records if r.lon is not None]
    frame = LocalFrame(float(np.mean(lats)), float(np.mean(lons)))

config = TruthBuilderConfig()  # defaults from the design doc: 60 s gap, 30 m truth sigma
truth = build_truth(records, frame, config)
print(frame)
for key, value in truth.stats.as_dict().items():
    print(f"{key:28s} {value}")

## 3. An hour of flights

Smoothed tracks in the local frame, sampled every 5 s. Range rings are every
50 km from the origin.

In [ ]:
fig, (ax_map, ax_alt) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.1, 1]})
colors = plt.cm.tab20(np.linspace(0, 1, max(len(truth), 1)))
for seg, color in zip(truth, colors):
    t = np.arange(seg.t[0], seg.t[-1], 5.0)
    pos, _, ok = seg.evaluate(t)
    label = f"{seg.callsign or seg.icao24} ({seg.segment_id})"
    ax_map.plot(pos[ok, 0] / 1e3, pos[ok, 1] / 1e3, color=color, lw=1.2, label=label)
    ax_map.plot(pos[ok, 0][-1] / 1e3, pos[ok, 1][-1] / 1e3, "o", color=color, ms=3)
    if seg.has_altitude:  # reported altitude, not ENU "up" (which bends with the earth)
        ax_alt.plot(
            (seg.t - start.timestamp()) / 60, seg.alt_m * 3.28084 / 100, color=color, lw=1.2
        )
for r_km in range(50, 201, 50):
    ax_map.add_patch(plt.Circle((0, 0), r_km, fill=False, ls=":", color="grey", lw=0.7))
ax_map.plot(0, 0, "k^")
ax_map.set(
    xlabel="east (km)",
    ylabel="north (km)",
    title=f"{len(truth)} segments, {truth.stats.aircraft} aircraft",
)
ax_map.set_aspect("equal")
ax_map.legend(fontsize=6, loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax_alt.set(
    xlabel=f"minutes after {start:%H:%M} UTC",
    ylabel="altitude (hundreds of ft)",
    title="Vertical profile (reported altitude)",
)
fig.tight_layout()

## 4. What the builder removed

One segment up close: the positions kept, the smoothed track, and any
position the speed gate rejected as a bad decode. (Duplicates from the
second receiver are dropped too, but they sit exactly on kept positions.)

In [ ]:
def outliers_of(seg):
    # Positions inside the segment's time span that the builder rejected, excluding duplicates.
    kept = set(zip(seg.lat, seg.lon))
    return [
        r
        for r in records
        if r.icao24 == seg.icao24
        and r.lat is not None
        and seg.t[0] <= r.ts.timestamp() <= seg.t[-1]
        and (r.lat, r.lon) not in kept
    ]


airborne = [s for s in truth if not s.on_ground]
seg = max(airborne, key=lambda s: (len(outliers_of(s)), len(s)))
bad = outliers_of(seg)
bad_e, bad_n, _ = frame.to_enu(
    [r.lat for r in bad],
    [r.lon for r in bad],
    np.interp([r.ts.timestamp() for r in bad], seg.t, seg.alt_m),
)

t = np.arange(seg.t[0], seg.t[-1], 1.0)
pos, vel, ok = seg.evaluate(t)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(seg.enu[:, 0] / 1e3, seg.enu[:, 1] / 1e3, ".", ms=3, label="kept positions")
ax1.plot(pos[:, 0] / 1e3, pos[:, 1] / 1e3, "-", lw=1, label="smoothed")
ax1.plot(bad_e / 1e3, bad_n / 1e3, "rx", ms=9, mew=2, label=f"rejected by speed gate ({len(bad)})")
ax1.set(
    xlabel="east (km)",
    ylabel="north (km)",
    title=f"{seg.callsign or seg.icao24}: {len(seg)} positions kept",
)
ax1.legend(fontsize=8)
minutes = (t - seg.t[0]) / 60
ax2.plot(minutes, np.hypot(vel[:, 0], vel[:, 1]) / 0.514444, label="spline derivative")
rv = seg.reported_velocity
ax2.plot(
    (seg.t - seg.t[0]) / 60,
    np.hypot(rv[:, 0], rv[:, 1]) / 0.514444,
    ".",
    ms=3,
    label="ADS-B velocity reports",
)
ax2.set(xlabel="minutes into segment", ylabel="ground speed (kt)", title="Speed")
ax2.legend(fontsize=8)
fig.tight_layout()

## 5. A second source for the same aircraft

Milestone 1 is done when the same aircraft from two sources agree within the
truth sigma (30 m by default). With `TRACKING_SIM_SECOND` set, the second
source is that file: another receiver's recording, or SBS exported from
another system (see `tools/README.md`). Its altitude is read as
`alt_unknown_ft`, because an exported file may not carry barometric altitude.
Without it, the notebook makes a **synthetic** stand-in: an independent
receiver with its own noise whose altitude also goes to `alt_unknown_ft`.

In [ ]:
if SECOND:
    second_name = SECOND
    # Whatever field the file used, treat its altitude as unknown until verified.
    second = [
        replace(r, alt_unknown_ft=r.altitude_ft, alt_baro_ft=None, alt_geom_ft=None)
        for r in open_source(SECOND, "second", altitude_field="alt_unknown_ft").replay(start, end)
    ]
else:
    from tracking_sim.truth.synthetic import ReceiverConfig, generate_traffic

    if "synthetic_1h" not in RECORDING:
        raise RuntimeError("set TRACKING_SIM_SECOND to compare your own recording")
    second_name = "synthetic stand-in"
    stand_in = generate_traffic(
        seed=1,
        # Same seed, so the same flights as the fixture, heard by a different receiver.
        receivers=(
            ReceiverConfig(
                "synthetic:second", position_sigma_m=12.0, altitude_field="alt_unknown_ft"
            ),
        ),
    )
    second = [r for r in stand_in.records if r.source_id == "synthetic:second"]

truth_b = build_truth(second, frame, config)
report = compare_truth(truth_b, truth)  # differences are recording minus second source
print(f"second source: {second_name}, {len(second)} records, {len(truth_b)} segments\n")
print(report.table())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(report.horizontal_m, bins=60, color="tab:blue")
ax1.axvline(report.sigma_m, color="k", ls="--", label=f"truth sigma {report.sigma_m:.0f} m")
ax1.set(xlabel="horizontal difference (m)", ylabel="samples", title="Same aircraft, same time")
ax1.legend()
ax2.hist(report.vertical_m * 3.28084, bins=60, color="tab:orange")
ax2.set(xlabel="vertical difference (ft)", title="Altitude: recording minus second source")
fig.tight_layout()

## 6. Is the second source's altitude barometric or geometric?

`assess_altitude_reference` compares the second source's altitude with the
recording's barometric altitude (and geometric, if the recording has it).
An SBS recording only carries barometric altitude, so a `baro` verdict here
can't rule out a geometric source on a day near standard pressure; see
`DECISIONS.md` (D10) for the full procedure.

In [ ]:
assessment = assess_altitude_reference(second, records, frame, config=config)
print(assessment.summary())

## 7. Stone Soup ground truth

Every segment exports as a Stone Soup `GroundTruthPath` with
`[x, vx, y, vy, z, vz]` states in the local frame, ready for the sensor
simulator and metrics in later milestones.

In [ ]:
paths = truth.ground_truth_paths(step_s=5.0)
path = paths[0]
print(f"{len(paths)} GroundTruthPaths; {path.id} has {len(path.states)} states")
state = path.states[len(path.states) // 2]
print(state.timestamp, np.asarray(state.state_vector).ravel().round(1))
print(state.metadata)